# v2.1-lite — Multi-Template SFT (Unsloth): Clinical Transcript -> Structured JSON

**Goal:** test whether adding supervised `mse` examples teaches the MSE ontology while preserving `soap` and `referral_a` performance, with `referral_b` kept as the held-out nearby-transfer check.

**Why Unsloth?** The standard bitsandbytes+peft stack OOMs on Kaggle T4 (14.56 GiB) at MAX_SEQ_LEN=1024. Unsloth's fused kernels + custom gradient checkpointing halve the VRAM footprint.

**What this proves**
- Qwen 2.5 3B Instruct + 4-bit + LoRA (Unsloth) can be fine-tuned on three template families in one run
- The model can be tested on whether supervised `mse` examples fix the ontology gap seen in `v2`
- Evidence spans remain grounded in the source transcript across all templates
- The training pipeline still scales cleanly to new templates without hardcoded per-template branches

**Eval design**
- Train: 150 samples (50 SOAP + 50 REFERRAL-A + 50 MSE), mixed and shuffled
- Eval in-dist: 30 samples (10 SOAP + 10 REFERRAL-A + 10 MSE) — same schemas as training
- Eval held-out: 10 samples (`referral_b`) — same referral schema semantics as `referral_a`, but unseen transcript style

**What this does NOT prove**
- Production transcript quality — transcripts are short synthetic dialogues
- Robust zero-shot transfer to a brand-new ontology — `mse` is no longer zero-shot in this run
- DPO/alignment improvements — deferred to v3
- Constrained decoding — deferred to v3

**Success criteria** (per template group):
- `json_parse_rate` >= 0.8
- `schema_keys_rate` >= 0.7
- `evidence_grounding_rate` >= 0.5
- `key_overlap_mean` >= 0.4

Main question: does supervised `mse` improve MSE extraction without materially hurting `soap`, `referral_a`, or held-out `referral_b` transfer?

In [ ]:
# Cell 1 — env check
import sys
print('Python:', sys.version)
!nvidia-smi

In [ ]:
# Cell 2 — clone repo via GITHUB_PAT secret
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

In [ ]:
# Cell 3 — install Unsloth (nightly) to fix transformers>=4.46 incompat, then local package
!pip install -q unsloth
!pip uninstall unsloth -y
!pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q -e .

import sys
sys.path.insert(0, '/kaggle/working/repo')

In [ ]:
# Cell 4 — load and mix training + eval data from repo
# Each record has: {"transcript": str, "template": str, <label_key>: dict}
import json, random
from src.data_generation.templates import REGISTRY

DATA_ROOT = 'data/qwen3.5_latest'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

# Training data: SOAP + REFERRAL-A + MSE mixed
train_rows = (load_jsonl(f'{DATA_ROOT}/train.soap.jsonl') +
              load_jsonl(f'{DATA_ROOT}/train.referral_a.jsonl') +
              load_jsonl(f'{DATA_ROOT}/train.mse.jsonl'))
random.Random(42).shuffle(train_rows)

# Eval in-distribution: SOAP + REFERRAL-A + MSE
eval_in_rows = (load_jsonl(f'{DATA_ROOT}/eval_in_dist.soap.jsonl') +
                load_jsonl(f'{DATA_ROOT}/eval_in_dist.referral_a.jsonl') +
                load_jsonl(f'{DATA_ROOT}/eval_in_dist.mse.jsonl'))

# Eval held-out: REFERRAL-B only
eval_zs_rows = load_jsonl(f'{DATA_ROOT}/eval_zero_shot.referral_b.jsonl')

print(f'train={len(train_rows)}  eval_in_dist={len(eval_in_rows)}  eval_held_out={len(eval_zs_rows)}')

# Sanity check: confirm all records have the template field
for split, rows in [('train', train_rows), ('eval_in', eval_in_rows), ('eval_zs', eval_zs_rows)]:
    missing = [i for i, r in enumerate(rows) if 'template' not in r]
    print(f'  {split}: template field missing in {len(missing)} records')

In [ ]:
# Cell 5 — load Qwen 2.5 3B Instruct, 4-bit via Unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024
MODEL_NAME  = 'unsloth/Qwen2.5-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,         # auto-detect bf16/fp16
    load_in_4bit=True,
)
print('loaded', MODEL_NAME)

In [ ]:
# Cell 6 — attach LoRA adapters via Unsloth
# use_gradient_checkpointing='unsloth' uses fused kernels — halves VRAM vs standard GC
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=0,
)
model.print_trainable_parameters()

In [ ]:
# Cell 7 — format data with chat template (template-aware)
# build_inference_messages constructs the system+user prompt for a given template,
# injecting the output spec so the model knows what JSON shape to produce.
from datasets import Dataset
from src.prompts import build_inference_messages

def to_text(row):
    template  = row['template']
    label_key = REGISTRY[template]['label_key']
    msgs = build_inference_messages(template, row['transcript'])
    msgs.append({
        'role': 'assistant',
        'content': json.dumps(row[label_key], ensure_ascii=False),
    })
    return tokenizer.apply_chat_template(msgs, tokenize=False)

train_ds   = Dataset.from_list([{'text': to_text(r)} for r in train_rows])
eval_in_ds = Dataset.from_list([{'text': to_text(r)} for r in eval_in_rows])

print(f'n_train={len(train_ds)}  n_eval_in_dist={len(eval_in_ds)}')
print('--- example 0 (first 600 chars) ---')
print(train_ds[0]['text'][:600])
lengths = [len(tokenizer.encode(d['text'])) for d in train_ds]
print(f'max={max(lengths)} mean={sum(lengths)/len(lengths):.0f} p95={sorted(lengths)[int(len(lengths)*0.95)]}')

In [ ]:
# Cell 8 — SFTTrainer config
from trl import SFTTrainer, SFTConfig

# === flip MAX_STEPS=4 (smoke test) -> 200 (real run) ===
MAX_STEPS  = 200
EVAL_EVERY = max(1, MAX_STEPS // 6)
print(f'MAX_STEPS={MAX_STEPS}  EVAL_EVERY={EVAL_EVERY}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_in_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=SFTConfig(
        output_dir='/kaggle/working/sft_out_v2_1_lite',
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=20,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy='steps',
        eval_steps=EVAL_EVERY,
        save_strategy='steps',
        save_steps=EVAL_EVERY,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=0,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        report_to='none',
    ),
)

In [ ]:
# Cell 9 — train
trainer.train()

## Eval loop

Four metrics computed **per template group** (in-dist vs held-out):

1. **json_parse_rate** — output is valid JSON
2. **schema_keys_rate** — top-level keys match the expected output spec
3. **key_overlap_mean** — Jaccard on the primary text field (chief_complaint for SOAP, reason for REFERRAL, appearance for MSE)
4. **evidence_grounding_rate** — fraction of `evidence_span` values that are verbatim substrings of the transcript

For `v2.1-lite`, `mse` is now in-distribution. The held-out check is `referral_b` only.

In [ ]:
# Cell 10 — eval loop (template-aware)
from src.prompts import build_inference_messages

FastLanguageModel.for_inference(model)  # Unsloth: enable optimised inference kernels


def generate(template: str, transcript: str) -> str:
    msgs = build_inference_messages(template, transcript)
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to('cuda')
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


def collect_spans(obj):
    """Recursively collect all evidence_span values from a nested dict/list."""
    spans = []
    if isinstance(obj, dict):
        if 'evidence_span' in obj:
            spans.append(obj['evidence_span'])
        for v in obj.values():
            spans.extend(collect_spans(v))
    elif isinstance(obj, list):
        for v in obj:
            spans.extend(collect_spans(v))
    return spans


# Per-template primary field used for key_overlap metric
PRIMARY_KEY = {
    'soap':       lambda o: o.get('subjective', {}).get('chief_complaint', {}).get('text', ''),
    'referral_a': lambda o: o.get('reason', {}).get('text', ''),
    'referral_b': lambda o: o.get('reason', {}).get('text', ''),
    'mse':        lambda o: o.get('appearance', {}).get('text', ''),
}

SCHEMA_KEYS = {
    'soap':       {'subjective', 'objective', 'assessment', 'plan'},
    'referral_a': {'specialty', 'patient', 'reason', 'history', 'current_meds', 'request'},
    'referral_b': {'specialty', 'patient', 'reason', 'history', 'current_meds', 'request'},
    'mse':        {'appearance', 'behaviour', 'speech', 'mood', 'affect', 'thought',
                   'cognition', 'insight'},
}


def score(pred_str: str, gold: dict, transcript: str, template: str) -> dict:
    try:
        pred = json.loads(pred_str)
    except Exception:
        return {'parse': 0, 'keys': 0, 'key_overlap': 0.0, 'ground': 0.0}

    keys = int(set(pred.keys()) >= SCHEMA_KEYS.get(template, set()))

    try:
        p_txt = PRIMARY_KEY[template](pred).lower()
        g_txt = PRIMARY_KEY[template](gold).lower()
        if not p_txt and not g_txt:
            overlap = 1.0          # both null/missing — perfect match
        elif not p_txt or not g_txt:
            overlap = 0.0          # one side missing — no overlap
        else:
            a, b    = set(p_txt.split()), set(g_txt.split())
            overlap = len(a & b) / max(len(a | b), 1)
    except Exception:
        overlap = 0.0

    spans  = [s for s in collect_spans(pred) if s]
    ground = sum(1 for s in spans if s.lower() in transcript.lower()) / max(len(spans), 1)

    return {'parse': 1, 'keys': keys, 'key_overlap': overlap, 'ground': ground}


def run_eval(rows, split_name):
    results_by_template = {}
    for row in rows:
        template  = row['template']
        label_key = REGISTRY[template]['label_key']
        pred_str  = generate(template, row['transcript'])
        s = score(pred_str, row[label_key], row['transcript'], template)
        results_by_template.setdefault(template, []).append(s)

    print(f'\n=== {split_name} ===')
    all_results = []
    for tmpl, res in sorted(results_by_template.items()):
        n = len(res)
        summary = {
            'template': tmpl,
            'n': n,
            'json_parse_rate':         sum(r['parse']       for r in res) / n,
            'schema_keys_rate':        sum(r['keys']        for r in res) / n,
            'key_overlap_mean':        sum(r['key_overlap'] for r in res) / n,
            'evidence_grounding_rate': sum(r['ground']      for r in res) / n,
        }
        print(json.dumps(summary, indent=2))
        all_results.extend(res)

    n = len(all_results)
    print(json.dumps({
        'split': split_name,
        'n_total': n,
        'json_parse_rate':         sum(r['parse']       for r in all_results) / n,
        'schema_keys_rate':        sum(r['keys']        for r in all_results) / n,
        'key_overlap_mean':        sum(r['key_overlap'] for r in all_results) / n,
        'evidence_grounding_rate': sum(r['ground']      for r in all_results) / n,
    }, indent=2))
    return results_by_template


in_dist_results   = run_eval(eval_in_rows, 'eval_in_dist')
held_out_results = run_eval(eval_zs_rows, 'eval_held_out')

In [ ]:
# Cell 11 — qualitative inspection: 1 example per template
seen = set()
all_eval = eval_in_rows + eval_zs_rows
for row in all_eval:
    t = row['template']
    if t in seen:
        continue
    seen.add(t)
    label_key = REGISTRY[t]['label_key']
    pred_str  = generate(t, row['transcript'])
    print('=' * 72)
    print(f'TEMPLATE: {t}')
    print('TRANSCRIPT:')
    print(row['transcript'])
    print('\nPREDICTED:')
    print(pred_str)
    print('\nGOLD:')
    print(json.dumps(row[label_key], indent=2))

In [ ]:
# Cell 12 — save LoRA adapter
ADAPTER_DIR = '/kaggle/working/adapter_v2_1_lite'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved ->', ADAPTER_DIR)
!ls -la $ADAPTER_DIR

In [ ]:
# Cell 13 — plot training curves + dump v2.1-lite logs
import matplotlib.pyplot as plt
from pathlib import Path

log     = trainer.state.log_history
out_dir = Path('/kaggle/working')
(out_dir / 'log_history_v2_1_lite.json').write_text(json.dumps(log, indent=2))

train_e = [e for e in log if 'loss' in e and 'eval_loss' not in e]
eval_e  = [e for e in log if 'eval_loss' in e]

ts  = [e['step']           for e in train_e]
tl  = [e['loss']           for e in train_e]
gn  = [e.get('grad_norm')  for e in train_e]
lr_ = [e.get('learning_rate') for e in train_e]
es  = [e['step']           for e in eval_e]
el  = [e['eval_loss']      for e in eval_e]

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0,0].plot(ts, tl, label='train')
if el: ax[0,0].plot(es, el, 'o-', label='eval')
ax[0,0].set_title('loss'); ax[0,0].set_xlabel('step'); ax[0,0].legend()
ax[0,1].plot(ts, gn); ax[0,1].set_title('grad_norm'); ax[0,1].set_xlabel('step')
ax[1,0].plot(ts, lr_); ax[1,0].set_title('learning_rate'); ax[1,0].set_xlabel('step')
ax[1,1].axis('off')
fig.tight_layout()
fig.savefig(out_dir / 'curves_train_v2_1_lite.png', dpi=120)
plt.show()
print('saved curves_train_v2_1_lite.png, log_history_v2_1_lite.json ->', out_dir)

## Export for local serving

Pipeline: LoRA adapter -> merge into base (fp16) -> convert HF -> GGUF (fp16) -> quantise -> Q4_K_M.

Output artefacts (download from `/kaggle/working/`):
- `qwen3b-v2_1_lite-multitemplate-f16.gguf` (~6 GB) — full precision
- `qwen3b-v2_1_lite-multitemplate-q4_k_m.gguf` (~2 GB) — quantised, for local llama.cpp serving

In [ ]:
# Cell A — merge LoRA adapter into base fp16 weights (CPU merge avoids VRAM spike)
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer
gc.collect(); torch.cuda.empty_cache()

base   = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu',
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
MERGED_DIR = '/kaggle/working/merged_fp16_v2_1_lite'
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print('merged ->', MERGED_DIR)
!ls -la $MERGED_DIR

In [ ]:
# Cell B — clone llama.cpp and convert HF -> GGUF (fp16)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/merged_fp16_v2_1_lite \
    --outfile /kaggle/working/qwen3b-v2_1_lite-multitemplate-f16.gguf \
    --outtype f16

In [ ]:
# Cell C — build llama-quantize and produce Q4_K_M
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF 2>&1 | tail -5
!cd /kaggle/working/llama.cpp && cmake --build build --config Release -j --target llama-quantize 2>&1 | tail -5
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/qwen3b-v2_1_lite-multitemplate-f16.gguf \
    /kaggle/working/qwen3b-v2_1_lite-multitemplate-q4_k_m.gguf \
    Q4_K_M
!ls -lh /kaggle/working/*.gguf

## v3 roadmap

1. **DPO preference dataset** — for each (transcript, template) pair, generate multiple completions and rank them by grounding quality + schema compliance to produce (chosen, rejected) pairs.
2. **DPO training** — fine-tune v2 adapter further with TRL DPOTrainer on preference pairs.
3. **Constrained decoding** — Outlines/jsonschema-guided generation to guarantee schema compliance at inference time.
4. **FastAPI server + Gradio demo** — template dropdown, SFT vs DPO toggle, live grounding highlighting.